# IBKR API notebook for fetching historical market data

#### Prerequisites: 
- ```pip install -r requirements.txt``` to install the necessary packages.

- Launch Trader Workstation (TWS) and enable ActiveX API (```File->Global Configuration->API->Settings``` then check ```Enable ActiveX and Socket Clients``` and uncheck ```Read-Only API```. Do not forget to apply the settings).

#### 1. Connection to IBKR API local gateway

```File->Global Configuration->API->Settings```
et cocher 
```Enable ActiveX and Socket Clients```
et décocher 
```Read-Only API```

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
import pandas as pd
from ib_async import *
util.startLoop()
global historical_data_interval, duration

ib = IB()
ib.connect('127.0.0.1', 7497, clientId=14)
if ib.isConnected():
    print("✅ Connected to IBKR API")
else:
    print("❌Failed to connect to IBKR API")
util.logToConsole(logging.INFO)

2025-08-26 13:03:16,634 - INFO - Connecting to 127.0.0.1:7497 with clientId 14...
2025-08-26 13:03:16,638 - INFO - Connected
2025-08-26 13:03:16,707 - INFO - Logged on to server version 178
2025-08-26 13:03:16,762 - INFO - Warning 2104, reqId -1: La connexion de donn\u00e9es de march\u00e9 est OK:usfuture
2025-08-26 13:03:16,764 - INFO - Warning 2104, reqId -1: La connexion de donn\u00e9es de march\u00e9 est OK:eufarm
2025-08-26 13:03:16,765 - INFO - Warning 2104, reqId -1: La connexion de donn\u00e9es de march\u00e9 est OK:cashfarm
2025-08-26 13:03:16,766 - INFO - Warning 2104, reqId -1: La connexion de donn\u00e9es de march\u00e9 est OK:eufarmnj
2025-08-26 13:03:16,767 - INFO - Warning 2104, reqId -1: La connexion de donn\u00e9es de march\u00e9 est OK:usfarm
2025-08-26 13:03:16,769 - INFO - Warning 2106, reqId -1: La connexion de donn\u00e9es HMDS est OK:euhmds
2025-08-26 13:03:16,770 - INFO - Warning 2106, reqId -1: La connexion de donn\u00e9es HMDS est OK:ushmds
2025-08-26 13:03:16

✅ Connected to IBKR API


2025-08-26 13:03:53,528 - INFO - Warning 2174, reqId 5: Avertissement: You submitted request with date-time attributes without explicit time zone. Please switch to use yyyymmdd-hh:mm:ss in UTC or use instrument time zone, like US/Eastern. Implied time zone functionality will be removed in the next API release, contract: Index(symbol='SPX', exchange='CBOE', currency='USD')
2025-08-26 13:38:12,265 - ERROR - Error 1100, reqId -1: La connexion entre IBKR et Trader Workstation a \u00e9t\u00e9 perdue.
2025-08-26 13:38:15,135 - INFO - Warning 2103, reqId -1: La connexion de donn\u00e9es de march\u00e9 est rompue:usfuture
2025-08-26 13:38:15,137 - INFO - Warning 2103, reqId -1: La connexion de donn\u00e9es de march\u00e9 est rompue:eufarm
2025-08-26 13:38:15,142 - INFO - Warning 2103, reqId -1: La connexion de donn\u00e9es de march\u00e9 est rompue:cashfarm
2025-08-26 13:38:15,146 - INFO - Warning 2103, reqId -1: La connexion de donn\u00e9es de march\u00e9 est rompue:eufarmnj
2025-08-26 13:38:

## Request Historical data

#### 2. Choose your contract

In [7]:
# Define the contract HERE
#contract = CFD('IBUST100', 'SMART', 'USD')
#contract = Forex(pair="EURUSD", exchange='IDEALPRO')
#contract = Stock(symbol='AAPL', exchange='SMART', currency='USD')
# contract = Index('NDX', 'NASDAQ', 'USD')
contract = Index('SPX', 'CBOE', 'USD')
#CAC 40
# contract = Index('FCHI', 'EUREX', 'EUR')


# Below => just some printing on contract chosen
contract_details = ib.reqContractDetails(contract)
# Extract and display the desired fields from contract_details
filtered_details = [
    {
        "secType": detail.contract.secType,
        "conId": detail.contract.conId,
        "symbol": detail.contract.symbol,
        "exchange": detail.contract.exchange,
        "longName": detail.longName,
        "timezoneId": detail.timeZoneId,
        "tradingHours": "\n".join(
            [f"  {segment}" for segment in detail.tradingHours.split(";")]
        ),
        "liquidHours": "\n".join(
            [f"  {segment}" for segment in detail.liquidHours.split(";")]
        ),
        "minSize": detail.minSize,
    }
    for detail in contract_details
]

# Print the filtered details in a clear format
for idx, detail in enumerate(filtered_details, start=1):
    print(f"Contract Detail {idx}:")
    for key, value in detail.items():
        print(f"  {key}: {value}")
    print()

Contract Detail 1:
  secType: IND
  conId: 416904
  symbol: SPX
  exchange: CBOE
  longName: S&P 500 Stock Index
  timezoneId: US/Central
  tradingHours:   20250826:0830-20250826:1500
  20250827:0830-20250827:1500
  20250828:0830-20250828:1500
  20250829:0830-20250829:1500
  liquidHours:   20250826:0830-20250826:1500
  20250827:0830-20250827:1500
  20250828:0830-20250828:1500
  20250829:0830-20250829:1500
  minSize: 1.0



#### (Optional) Check first data timestamp available

In [8]:
timestamp = ib.reqHeadTimeStamp(contract, whatToShow='TRADES', useRTH=False)
formatted_time = timestamp.strftime('%B %d, %Y, %H:%M')
logging.info(f"First date of data available: {formatted_time}")

2025-08-26 16:19:37,599 - INFO - First date of data available: March 04, 2004, 14:30


#### 3. End date choice for data request

In [9]:
# yesterday's date
end_date = (pd.Timestamp.now(tz='UTC') - pd.DateOffset(days=1)).strftime('%Y%m%d %H:%M:%S')

In [ ]:
# today's date
end_date = pd.Timestamp.now(tz='UTC').strftime('%Y%m%d-%H:%M:%S')

In [ ]:
# custom end date
# format expected : YYYYMMDD HH:MM:SS
end_date = '20221125 22:00:00'

#### 4. Main loop for fetching historical data

Args
- **contract**: Contract of interest.  
- **endDateTime**:  
    - Can be set to `''` to indicate the current time.  
    - Can be given as a `datetime.date` or `datetime.datetime`.  
    - Can be given as a string in `'yyyyMMdd HH:mm:ss'` format.  
    - If no timezone is given, the TWS login timezone is used.  
- **durationStr**: Time span of all the bars. Examples:  
    - `'60 S'`, `'30 D'`, `'13 W'`, `'6 M'`, `'10 Y'`.  
- **barSizeSetting**: Time period of one bar. Must be one of:  
    - `'1 secs'`, `'5 secs'`, `'10 secs'`, `'15 secs'`, `'30 secs'`,  
    - `'1 min'`, `'2 mins'`, `'3 mins'`, `'5 mins'`, `'10 mins'`, `'15 mins'`,  
    - `'20 mins'`, `'30 mins'`,  
    - `'1 hour'`, `'2 hours'`, `'3 hours'`, `'4 hours'`, `'8 hours'`,  
    - `'1 day'`, `'1 week'`, `'1 month'`.  
- **whatToShow**: Specifies the source for constructing bars. Must be one of:  
    - `'TRADES'`, `'MIDPOINT'`, `'BID'`, `'ASK'`, `'BID_ASK'`,  
    - `'ADJUSTED_LAST'`, `'HISTORICAL_VOLATILITY'`, `'OPTION_IMPLIED_VOLATILITY'`,  
    - `'REBATE_RATE'`, `'FEE_RATE'`, `'YIELD_BID'`, `'YIELD_ASK'`, `'YIELD_BID_ASK'`, `'YIELD_LAST'`.  
    - For `'SCHEDULE'`, use `:meth:.reqHistoricalSchedule`.  
- **useRTH**:  
    - If `True`, only show data from within Regular Trading Hours.  
    - If `False`, show all data.  
- **formatDate**:  
    - For an intraday request, setting to `2` will cause the returned date fields to be timezone-aware `datetime.datetime` with UTC timezone, instead of local timezone as used by TWS.  
- **keepUpToDate**:  
    - If `True`, a realtime subscription is started to keep the bars updated.  
    - `endDateTime` must be set empty (`''`) then.  
- **chartOptions**: Unknown.  
- **timeout**:  
    - Timeout in seconds after which to cancel the request and return an empty bar series. 
    - If the data request is huge, this parameter could spoil the request  
    - Set to `0` to wait indefinitely.  

In [10]:
historical_data_interval = '10 secs' # Candle period to fetch
request_duration = '6 M'  # Duration in days (use D, not "day"). Use a very big value if you want the maximum historical data, it will fetch the maximum available automatically.
price_source = 'TRADES'  # 'BID', 'ASK', or 'TRADES' (note that for some symbols, (e.g. EURUSD) only 'BID' and 'ASK' are available)

bars = ib.reqHistoricalData(
        contract,
        endDateTime=end_date,
        durationStr=str(request_duration),
        barSizeSetting=str(historical_data_interval),
        whatToShow=price_source,
        useRTH=False, 
        formatDate=2,
        timeout = 0)

bars[0]
new_df = util.df(bars)

display(new_df.head())
display(new_df.tail())
# Remove the 'volume', 'average', and 'barCount' columns from the DataFrame
new_df = new_df.drop(columns=['volume', 'average', 'barCount'])

# Display the updated DataFrame
new_df.head()

# Nouvelle cellule pour le premier téléchargement de données (pas de fichier existant)

import os

# save_path doit être défini comme dans la cellule précédente
# save_path = f"../marketData/{contract.symbol}_10secs_20240915_to_20250826_{price_source}.csv"

save_path = f"../marketData/{contract.symbol}_10secs_20240915_to_20250826_{price_source}.csv" # Here, enter the correct file path for the new csv data file

# Sauvegarder le DataFrame nouvellement téléchargé
new_df.to_csv(save_path, index=True)
print(f"New data saved to: {save_path}")

# Vérification optionnelle
import pandas as pd
df_check = pd.read_csv(save_path, index_col=0)
display(df_check.head())
display(df_check.tail())

,date,open,high,low,close,volume,average,barCount
0,2025-02-24 14:30:00+00:00,6026.69,6031.45,6026.69,6030.94,0.0,0.0,9
1,2025-02-24 14:30:10+00:00,6030.37,6031.56,6030.37,6031.13,0.0,0.0,10
2,2025-02-24 14:30:20+00:00,6031.66,6033.08,6031.66,6033.08,0.0,0.0,10
3,2025-02-24 14:30:30+00:00,6032.81,6033.95,6032.51,6033.23,0.0,0.0,10
4,2025-02-24 14:30:40+00:00,6032.86,6035.42,6032.69,6035.42,0.0,0.0,10


,date,open,high,low,close,volume,average,barCount
293755,2025-08-22 19:59:10+00:00,6463.85,6465.02,6463.85,6464.66,0.0,0.0,10
293756,2025-08-22 19:59:20+00:00,6464.69,6465.55,6464.66,6465.50,0.0,0.0,10
293757,2025-08-22 19:59:30+00:00,6465.62,6467.48,6465.42,6467.48,0.0,0.0,10
293758,2025-08-22 19:59:40+00:00,6467.53,6467.83,6467.09,6467.32,0.0,0.0,10
293759,2025-08-22 19:59:50+00:00,6467.62,6467.62,6465.40,6466.35,0.0,0.0,10


New data saved to: ../marketData/SPX_10secs_20240915_to_20250826_TRADES.csv


,date,open,high,low,close
0,2025-02-24 14:30:00+00:00,6026.69,6031.45,6026.69,6030.94
1,2025-02-24 14:30:10+00:00,6030.37,6031.56,6030.37,6031.13
2,2025-02-24 14:30:20+00:00,6031.66,6033.08,6031.66,6033.08
3,2025-02-24 14:30:30+00:00,6032.81,6033.95,6032.51,6033.23
4,2025-02-24 14:30:40+00:00,6032.86,6035.42,6032.69,6035.42


,date,open,high,low,close
293755,2025-08-22 19:59:10+00:00,6463.85,6465.02,6463.85,6464.66
293756,2025-08-22 19:59:20+00:00,6464.69,6465.55,6464.66,6465.50
293757,2025-08-22 19:59:30+00:00,6465.62,6467.48,6465.42,6467.48
293758,2025-08-22 19:59:40+00:00,6467.53,6467.83,6467.09,6467.32
293759,2025-08-22 19:59:50+00:00,6467.62,6467.62,6465.40,6466.35


#### 5. Convert the list of bars to a data frame, print the first / last rows and remove useless columns:

In [6]:
bars[0]
new_df = util.df(bars)

display(new_df.head())
display(new_df.tail())
# Remove the 'volume', 'average', and 'barCount' columns from the DataFrame
new_df = new_df.drop(columns=['volume', 'average', 'barCount'])

# Display the updated DataFrame
new_df.head()

,date,open,high,low,close,volume,average,barCount
0,2025-08-20 13:30:00+00:00,6406.62,6408.40,6406.62,6408.40,0.0,0.0,9
1,2025-08-20 13:30:10+00:00,6407.92,6407.92,6406.22,6406.22,0.0,0.0,10
2,2025-08-20 13:30:20+00:00,6405.99,6407.20,6405.51,6406.19,0.0,0.0,9
3,2025-08-20 13:30:30+00:00,6405.80,6406.15,6404.87,6404.87,0.0,0.0,9
4,2025-08-20 13:30:40+00:00,6404.64,6405.50,6404.35,6404.35,0.0,0.0,10


,date,open,high,low,close,volume,average,barCount
7015,2025-08-22 19:59:10+00:00,6463.85,6465.02,6463.85,6464.66,0.0,0.0,10
7016,2025-08-22 19:59:20+00:00,6464.69,6465.55,6464.66,6465.50,0.0,0.0,10
7017,2025-08-22 19:59:30+00:00,6465.62,6467.48,6465.42,6467.48,0.0,0.0,10
7018,2025-08-22 19:59:40+00:00,6467.53,6467.83,6467.09,6467.32,0.0,0.0,10
7019,2025-08-22 19:59:50+00:00,6467.62,6467.62,6465.40,6466.35,0.0,0.0,10


,date,open,high,low,close
0,2025-08-20 13:30:00+00:00,6406.62,6408.40,6406.62,6408.40
1,2025-08-20 13:30:10+00:00,6407.92,6407.92,6406.22,6406.22
2,2025-08-20 13:30:20+00:00,6405.99,6407.20,6405.51,6406.19
3,2025-08-20 13:30:30+00:00,6405.80,6406.15,6404.87,6404.87
4,2025-08-20 13:30:40+00:00,6404.64,6405.50,6404.35,6404.35


Now, you have a **pandas DataFrame** containing the historical data for your chosen contract. You can use this DataFrame for further analysis or visualization as needed. 
In the next cell, you can update an existing csv file by merging it with the new data. 

## Additional features

#### DataFrame update (only for existing data)
Update the dataframe by merging the new data with old ones

**IMPORTANT**: Use the correct format file : `../marketData/{contract.symbol}_{candle_period}_{first_date}_to_{last_date}_{price_source}.csv`

Date format to use: `YYYYMMDD`

In [ ]:
import pandas as pd
from Helpers import merge_ohlc_dataframes
import os

# Load your existing data - use index_col=0 to treat first column as index
existing_file_path = "../marketData/NDX_10secs_20220214_to_20250716_TRADES.csv" # Here, enter the correct file path fo the existing csv data file
existing_df = pd.read_csv(existing_file_path, index_col=0)
display(existing_df.head())

# Merge the dataframes
merged_df = merge_ohlc_dataframes(existing_df, new_df, frequency='10s') # Adjust frequency as needed
display(merged_df.head())
display(merged_df.tail())

# Save the merged dataframe
save_path = f"../marketData/{contract.symbol}_10secs_20240915_to_20250826_{price_source}.csv" # Here, enter the correct file path for the new csv data file
merged_df.to_csv(save_path, index=True)
print(f"Merged data saved to: {save_path}")
# Delete the original file if needed
if os.path.exists(existing_file_path):
    os.remove(existing_file_path)
    print(f"Deleted original file: {existing_file_path}")


FileNotFoundError: [Errno 2] No such file or directory: '../marketData/NDX_10secs_20220214_to_20250716_TRADES.csv'

In [8]:
# Nouvelle cellule pour le premier téléchargement de données (pas de fichier existant)

import os

# save_path doit être défini comme dans la cellule précédente
# save_path = f"../marketData/{contract.symbol}_10secs_20240915_to_20250826_{price_source}.csv"

save_path = f"../marketData/{contract.symbol}_10secs_20240915_to_20250826_{price_source}.csv" # Here, enter the correct file path for the new csv data file

# Sauvegarder le DataFrame nouvellement téléchargé
new_df.to_csv(save_path, index=True)
print(f"New data saved to: {save_path}")

# Vérification optionnelle
import pandas as pd
df_check = pd.read_csv(save_path, index_col=0)
display(df_check.head())
display(df_check.tail())

New data saved to: ../marketData/SPX_10secs_20240915_to_20250826_TRADES.csv


,date,open,high,low,close
0,2025-08-20 13:30:00+00:00,6406.62,6408.40,6406.62,6408.40
1,2025-08-20 13:30:10+00:00,6407.92,6407.92,6406.22,6406.22
2,2025-08-20 13:30:20+00:00,6405.99,6407.20,6405.51,6406.19
3,2025-08-20 13:30:30+00:00,6405.80,6406.15,6404.87,6404.87
4,2025-08-20 13:30:40+00:00,6404.64,6405.50,6404.35,6404.35


,date,open,high,low,close
7015,2025-08-22 19:59:10+00:00,6463.85,6465.02,6463.85,6464.66
7016,2025-08-22 19:59:20+00:00,6464.69,6465.55,6464.66,6465.50
7017,2025-08-22 19:59:30+00:00,6465.62,6467.48,6465.42,6467.48
7018,2025-08-22 19:59:40+00:00,6467.53,6467.83,6467.09,6467.32
7019,2025-08-22 19:59:50+00:00,6467.62,6467.62,6465.40,6466.35


#### Checking data integrity

In [ ]:
from Helpers import checkDataFile, visualize_data_gaps
import matplotlib.pyplot as plt

symbol = 'EUR'
interval = '10secs'
start_date = '20240915'
end_date = '20250717'
price_source= 'ASK'

# 1. Load the existing dataframe
save_path = f"../marketData/{symbol}_{interval}_{start_date}_to_{end_date}_{price_source}.csv"

# Analyze data gaps
report = checkDataFile(
    file_path=save_path, 
    interval=interval
)

# Print summary
print(f"Analyzed {report['total_trading_days']} trading days")
print(f"Found {report['days_with_gaps']} days with gaps ({report['analysis_summary']['gap_percentage']:.2f}%)")
print(f"Total gaps detected: {report['total_gaps']}")

# Visualize the gaps
fig = visualize_data_gaps(report)
plt.show()

# To examine specific days with large gaps
problem_days = {date: data for date, data in report["gaps_by_date"].items() 
                if data["missing_points"] > 10}
print(f"Days with more than 10 missing points: {len(problem_days)}")
for date, data in sorted(problem_days.items()):
    print(f"{date}: Missing {data['missing_points']} of {data['expected_points']} points")